# 01 — Turn Indicator Detection

Use `TurnIndicator` to automatically detect left and right turns from `GYR_Y_DEG`.
Explore threshold sensitivity and visualize flagged events on the raw signal.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from scooter_imu.data.loader import load_raw
from scooter_imu.data.preprocessor import preprocess
from scooter_imu.algorithms.turn_indicator import TurnIndicator

RAW_PATH = Path("../data/raw/ScooterIMUData.csv")

In [2]:
df = preprocess(load_raw(RAW_PATH))
result = TurnIndicator().predict(df)
result[["timestamp", "gyr_y", "turn_direction", "turn_level"]].head(20)

,timestamp,gyr_y,turn_direction,turn_level
0,2024-03-15 10:00:00.000,2.451354,Left,2
1,2024-03-15 10:00:00.020,-3.706539,Right,2
2,2024-03-15 10:00:00.040,-2.359132,Right,2
3,2024-03-15 10:00:00.060,0.398092,Left,1
4,2024-03-15 10:00:00.080,1.306252,Left,2
5,2024-03-15 10:00:00.100,0.050222,Left,1
6,2024-03-15 10:00:00.120,-0.263588,Right,1
7,2024-03-15 10:00:00.140,1.796928,Left,2
8,2024-03-15 10:00:00.160,6.624943,Left,2
9,2024-03-15 10:00:00.180,-2.241318,Right,2


In [3]:
# Summary of detected events
print(result["turn_direction"].value_counts())
print(result["turn_level"].value_counts())

turn_direction
Left     1572
Right    1424
Name: count, dtype: int64
turn_level
2    2212
1     784
0       4
Name: count, dtype: int64


In [4]:
# Visualise GYR_Y with turn events overlaid
fig = go.Figure()
fig.add_trace(go.Scatter(x=result["timestamp"], y=result["gyr_y"],
                         name="GYR_Y", line=dict(color="gray", width=1)))

for direction, colour in [("Left", "blue"), ("Right", "red")]:
    mask = result["turn_direction"] == direction
    fig.add_trace(go.Scatter(
        x=result.loc[mask, "timestamp"], y=result.loc[mask, "gyr_y"],
        mode="markers", name=direction, marker=dict(color=colour, size=4)))

fig.update_layout(title="Turn Events on GYR_Y", xaxis_title="Time", yaxis_title="deg/s")
fig.show()

In [5]:
# Threshold sensitivity: vary dead_band and count events
rows = []
for db in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]:
    r = TurnIndicator(dead_band=db).predict(df)
    rows.append({"dead_band": db, "turn_rows": (r["turn_level"] > 0).sum()})
pd.DataFrame(rows)

,dead_band,turn_rows
0,0.001,2999
1,0.005,2996
2,0.010,2993
3,0.050,2965
4,0.100,2911
5,0.500,2593


## Scooter Indicator Schematic

Top-down view of the scooter showing which turn signal is active.
Yellow glowing triangle = active blinker. The three states are shown below.

In [ ]:
import plotly.graph_objects as go

def _indicator_fig(turn_direction):
    """Top-down scooter schematic. turn_direction: 'Left' | 'Right' | None"""
    left_on  = turn_direction == "Left"
    right_on = turn_direction == "Right"
    lc = "#FFD700" if left_on  else "#333333"
    rc = "#FFD700" if right_on else "#333333"
    gl = "rgba(255,215,0,0.25)" if left_on  else "rgba(0,0,0,0)"
    gr = "rgba(255,215,0,0.25)" if right_on else "rgba(0,0,0,0)"

    shapes = [
        # road background
        dict(type="rect",   x0=-2.5, y0=-2.5, x1=2.5,  y1=2.5,  fillcolor="#555", line_width=0, layer="below"),
        dict(type="line",   x0=-2.5, y0=-0.8, x1=2.5,  y1=-0.8, line=dict(color="#888", width=1, dash="dash")),
        dict(type="line",   x0=-2.5, y0=0.8,  x1=2.5,  y1=0.8,  line=dict(color="#888", width=1, dash="dash")),
        dict(type="line",   x0=-2.5, y0=0,    x1=2.5,  y1=0,    line=dict(color="#fff", width=1, dash="dot")),
        # scooter body
        dict(type="rect",   x0=-0.28, y0=-0.95, x1=0.28,  y1=1.15, fillcolor="#1e6fa5", line=dict(color="#0d4f7a", width=2)),
        dict(type="rect",   x0=-0.22, y0=-0.30, x1=0.22,  y1=0.35, fillcolor="#3a3a3a", line=dict(color="#222",    width=1)),
        dict(type="circle", x0=-0.12, y0=0.88,  x1=0.12,  y1=1.08, fillcolor="#ffe066", line=dict(color="#ccaa00", width=1)),
        dict(type="rect",   x0=-0.13, y0=0.72,  x1=0.13,  y1=1.10, fillcolor="#111",    line=dict(color="#333",    width=1)),
        dict(type="rect",   x0=-0.13, y0=-1.10, x1=0.13,  y1=-0.72, fillcolor="#111",   line=dict(color="#333",    width=1)),
        dict(type="line",   x0=-0.55, y0=0.62,  x1=0.55,  y1=0.62, line=dict(color="#aaa", width=5)),
        dict(type="circle", x0=-0.18, y0=0.35,  x1=0.18,  y1=0.65, fillcolor="#e8c87a", line=dict(color="#888",    width=1)),
        # signal glow halos
        dict(type="circle", x0=-1.20, y0=0.25, x1=-0.70, y1=0.75, fillcolor=gl, line_width=0),
        dict(type="circle", x0= 0.70, y0=0.25, x1= 1.20, y1=0.75, fillcolor=gr, line_width=0),
        # turn signal triangles
        dict(type="path", path="M -0.55 0.7 L -1.05 0.5 L -0.55 0.3 Z", fillcolor=lc, line=dict(color=lc, width=1)),
        dict(type="path", path="M  0.55 0.7 L  1.05 0.5 L  0.55 0.3 Z", fillcolor=rc, line=dict(color=rc, width=1)),
    ]

    label = ("◄ LEFT INDICATOR ON" if left_on else
             "RIGHT INDICATOR ON ►" if right_on else "GOING STRAIGHT")
    label_col = "#FFD700" if (left_on or right_on) else "#aaddff"

    fig = go.Figure()
    fig.update_layout(
        shapes=shapes,
        annotations=[dict(x=0, y=-1.75, text=f"<b>{label}</b>", showarrow=False,
                          font=dict(color=label_col, size=13, family="monospace"),
                          xref="x", yref="y")],
        width=370, height=430,
        paper_bgcolor="#1e1e2e", plot_bgcolor="#555",
        xaxis=dict(visible=False, range=[-2.5, 2.5]),
        yaxis=dict(visible=False, range=[-2.2, 1.6]),
        margin=dict(l=5, r=5, t=45, b=5),
        title=dict(text="Turn Signal Status", font=dict(color="white", size=15), x=0.5),
    )
    return fig

for state in [None, "Left", "Right"]:
    _indicator_fig(state).show()

## Trip Replay — Actual Detected Events

Steps through every turn event detected in the real trip data.
Each frame updates the schematic and shows the timestamp and signal level.
Press **Play** or drag the slider to replay the trip.

In [ ]:
import numpy as np
import plotly.graph_objects as go

events  = result[result["turn_level"] > 0].reset_index(drop=True)
sample_idx = np.linspace(0, len(events) - 1, min(60, len(events)), dtype=int)
sampled    = events.iloc[sample_idx].reset_index(drop=True)

def _shapes_for(turn_direction):
    lc = "#FFD700" if turn_direction == "Left"  else "#333"
    rc = "#FFD700" if turn_direction == "Right" else "#333"
    gl = "rgba(255,215,0,0.2)" if turn_direction == "Left"  else "rgba(0,0,0,0)"
    gr = "rgba(255,215,0,0.2)" if turn_direction == "Right" else "rgba(0,0,0,0)"
    return [
        dict(type="rect",   x0=-2.5, y0=-2.5, x1=2.5, y1=2.5, fillcolor="#555", line_width=0, layer="below"),
        dict(type="line",   x0=-2.5, y0=0,    x1=2.5, y1=0,   line=dict(color="#fff", width=1, dash="dot")),
        dict(type="rect",   x0=-0.28, y0=-0.95, x1=0.28,  y1=1.15, fillcolor="#1e6fa5", line=dict(color="#0d4f7a", width=2)),
        dict(type="rect",   x0=-0.13, y0=0.72,  x1=0.13,  y1=1.10, fillcolor="#111",    line=dict(color="#333",    width=1)),
        dict(type="rect",   x0=-0.13, y0=-1.10, x1=0.13,  y1=-0.72, fillcolor="#111",   line=dict(color="#333",    width=1)),
        dict(type="circle", x0=-0.12, y0=0.88,  x1=0.12,  y1=1.08, fillcolor="#ffe066", line=dict(color="#ccaa00", width=1)),
        dict(type="line",   x0=-0.55, y0=0.62,  x1=0.55,  y1=0.62, line=dict(color="#aaa", width=5)),
        dict(type="circle", x0=-0.18, y0=0.35,  x1=0.18,  y1=0.65, fillcolor="#e8c87a", line=dict(color="#888",    width=1)),
        dict(type="circle", x0=-1.20, y0=0.25, x1=-0.70, y1=0.75, fillcolor=gl, line_width=0),
        dict(type="circle", x0= 0.70, y0=0.25, x1= 1.20, y1=0.75, fillcolor=gr, line_width=0),
        dict(type="path", path="M -0.55 0.7 L -1.05 0.5 L -0.55 0.3 Z", fillcolor=lc, line=dict(color=lc, width=1)),
        dict(type="path", path="M  0.55 0.7 L  1.05 0.5 L  0.55 0.3 Z", fillcolor=rc, line=dict(color=rc, width=1)),
    ]

frames = []
for _, row in sampled.iterrows():
    d  = row["turn_direction"] if str(row["turn_direction"]) != "<NA>" else "None"
    ts = str(row["timestamp"])[:19]
    label_text = f"{'◄ LEFT' if d=='Left' else 'RIGHT ►' if d=='Right' else 'STRAIGHT'} | Level {int(row['turn_level'])} | {ts}"
    label_col  = "#FFD700" if d in ("Left", "Right") else "#aaddff"
    frames.append(go.Frame(
        layout=dict(
            shapes=_shapes_for(d),
            annotations=[dict(x=0, y=-1.75, text=f"<b>{label_text}</b>", showarrow=False,
                              font=dict(color=label_col, size=11, family="monospace"),
                              xref="x", yref="y")],
        ),
        name=str(_),
    ))

fig = go.Figure(
    layout=go.Layout(
        shapes=_shapes_for("None"),
        annotations=[dict(x=0, y=-1.75, text="<b>Press ▶ Play to replay turn events</b>",
                          showarrow=False, font=dict(color="#aaddff", size=12, family="monospace"),
                          xref="x", yref="y")],
        width=420, height=465,
        paper_bgcolor="#1e1e2e", plot_bgcolor="#555",
        xaxis=dict(visible=False, range=[-2.5, 2.5]),
        yaxis=dict(visible=False, range=[-2.2, 1.6]),
        margin=dict(l=5, r=5, t=50, b=10),
        title=dict(text="Turn Events — Trip Replay", font=dict(color="white", size=15), x=0.5),
        updatemenus=[dict(
            type="buttons", showactive=False, y=-0.12, x=0.5, xanchor="center",
            buttons=[
                dict(label="▶  Play",  method="animate",
                     args=[None, dict(frame=dict(duration=600, redraw=True), fromcurrent=True)]),
                dict(label="⏸  Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0), mode="immediate")]),
            ],
        )],
        sliders=[dict(
            steps=[dict(args=[[f.name], dict(frame=dict(duration=0), mode="immediate")],
                        method="animate", label="") for f in frames],
            active=0, y=-0.04, len=0.9, x=0.05, currentvalue=dict(visible=False),
        )],
    ),
    frames=frames,
)
fig.show()